In [1]:
import os
import glob
import tgt
import pandas as pd
import pickle
import tqdm
import warnings
warnings.simplefilter("ignore")


import sys
sys.path.insert(0, "../../..")

from utils import data_processing
DATASET_PATH = "/Volumes/Transcend/USC-TIMIT/EMA/"

In [2]:
preferred_ipas_f = "../../pronunciation_counts/preferred_ipas_ae.pkl"

In [3]:
trans_fs = glob.glob(os.path.join(DATASET_PATH, "Data/*/trans/*.trans"))

In [4]:
def get_wav_path(trans_f):
    identifier = os.path.basename(trans_f).replace(".trans", "")
    subject = identifier.split("_")[2]
    return f"/Volumes/Transcend/USC-TIMIT/EMA/Data/{subject.upper()}/wav/{identifier}.wav"

def get_mat_path(trans_f):
    identifier = os.path.basename(trans_f).replace(".trans", "")
    subject = identifier.split("_")[2]
    return f"/Volumes/Transcend/USC-TIMIT/EMA/Data/{subject.upper()}/mat/{identifier}.mat"

In [5]:
trans_f = trans_fs[0]
df = pd.read_csv(trans_f, header=None)

In [6]:
dfs_final = []

for trans_f in tqdm.tqdm(trans_fs):
    art_f = get_mat_path(trans_f)
    ac_f = get_wav_path(trans_f)
    if not os.path.exists(art_f) or not os.path.exists(ac_f):
        continue
    
    df = pd.read_csv(trans_f, header=None)
    df_final = data_processing.alignment_to_dataframe(
        df,
        lang="ae",
        preferred_ipas_f=preferred_ipas_f,
    )

    df_final["art_f"] = art_f
    df_final["ac_f"] = ac_f
    dfs_final.append(df_final)

  0%|          | 0/370 [00:00<?, ?it/s]

100%|██████████| 370/370 [03:51<00:00,  1.60it/s]


In [7]:
df = pd.concat(dfs_final, ignore_index=True)

In [8]:
df.to_csv("datasets/processed_usema.csv", index=False)